<a href="https://colab.research.google.com/github/jcmachicao/cur_MMAI__multimodal/blob/main/cur_IAMM___Evaluacion_Sesion4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Evaluación de modelos multimodales**
**Hilo conductor:**
* Una métrica global no dice *qué* falla.
* Pero clasifica los errores
* Esa clasificación es una ontología que nace de la práctica

| Parte | Corpus | Capa evaluada | Pregunta |
|---|---|---|---|
| A | Bicicletas (`textos_multimodales.json`) | 1 · Representacional | ¿El espacio vectorial recupera mis categorías? ¿Son buenas mis categorías? |
| B | Caso policial (modelo de mundo + corpus) | 2 · Ontológica/epistemológica y 3 · Funcional | ¿El sistema afirma solo lo que la evidencia permite afirmar? |

Nota: Ajustar las rutas en la celda siguiente. En Colab monta Drive; fuera de Colab lee la carpeta actual.

In [1]:
import os, re, json, time
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print

try:
    from google.colab import drive
    drive.mount("/content/drive")
    RUTA_BICI = "drive/My Drive/00 data/bike img/"
    RUTA_CASO = "drive/My Drive/00 data/caso/"
except ImportError:
    RUTA_BICI = RUTA_CASO = "./"

Mounted at /content/drive


## PARTE A · Capa 1: evaluar el espacio de representación (corpus de bicicletas)

Tu corpus ya trae una verdad de referencia: `topic` (9 categorías) y `type` (3). Hasta ahora solo servían para pintar. Aquí las usamos para **medir**:

* **P@k**: de los k vecinos más cercanos de cada ítem, qué fracción comparte su etiqueta.
* **ARI**: concordancia entre los clusters de KMeans y las etiquetas (0 ≈ azar, 1 = perfecto).

Se calculan para `topic` **y** para `type`. Si el espacio se parece más a `type` que a `topic`, está agrupando por *registro* (reglamento vs. descripción visual) y no por *tema*.

In [2]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

with open(RUTA_BICI + "textos_multimodales.json", encoding="utf-8") as f:
    df = pd.DataFrame(json.load(f)["items"])

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34 entries, 0 to 33
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        34 non-null     object
 1   source    34 non-null     object
 2   type      34 non-null     object
 3   topic     34 non-null     object
 4   title     34 non-null     object
 5   text      34 non-null     object
 6   keywords  34 non-null     object
dtypes: object(7)
memory usage: 2.0+ KB


In [4]:
df.loc[0]

,0
id,img_01
source,imagen
type,descripcion_visual
topic,accidente_interseccion
title,Accidente urbano
text,Ciclista atropellado en vía urbana con tráfico...
keywords,"[accidente, ciclista, tráfico urbano, colisión..."


In [5]:
print(len(df), "ítems |", df.topic.nunique(), "topics |", df.type.nunique(), "types")
display(pd.crosstab(df.topic, df.type, margins=True))

34 ítems | 9 topics | 3 types


type,descripcion_visual,reglamento,reporte,All
topic,,,,
accidente_interseccion,3,1,1,5
control_municipal,2,0,1,3
convivencia_vial,2,1,2,5
infraestructura_ciclovia,3,1,0,4
mantenimiento_urbano,1,1,1,3
movilidad_recreativa,1,2,1,4
riesgo_vehicular,1,1,1,3
seguridad_nocturna,0,1,2,3
seguridad_preventiva,1,2,1,4


La función evaluar_espacio evalúa con qué **efectividad** un espacio vectorial captura las categorías inherentes ("topic" y "type") de la data.

Lo hace calculando dos **métricas**:

> *  P@k (precisión en k): Esta métrica verifica, para cada item, qué fracción de sus `k` vecinos cercanos comparten el mimsmo `topic` y el mismo `type`. Si el espacio representa bien las categorías, aquellos items con las mismas categorías deberían estar cerca en ese espacio, y esto diría que sus vecinos van a mantener sus etiquetas.
>* ARI (Adjusted Rand Index): Esta métrica evalúa que tan bien los clusters no supervisados (generados por un algoritmo como K-Means) se alinean con sus `topic` y `type` definidos. Esto informa sobre el nivel de acuerdo entre los clusters encontrados y las categorias conocidas ancladas (grounding).

Esencialmente, para cada espacio vectorial que se forma (vectores aleatorios, TF-IDF, o Embeddings con Sentence Transformers), la función calcula las métricas para `topic` y `type`.

Esto ayuda a comprender si el espacio vectorial está agrupando los textos basados en un tema real (`topic`) o por su formato/estilo (`type`).


In [6]:
def evaluar_espacio(E, nombre, vecinos=(1, 3), ks=(6, 9), seed=42):
    E = np.asarray(E)
    S = cosine_similarity(E); np.fill_diagonal(S, -1)
    orden = np.argsort(-S, axis=1)
    fila = {"espacio": nombre}
    for etiqueta in ("topic", "type"):
        y = df[etiqueta].to_numpy()
        for k in vecinos:
            fila[f"P@{k} {etiqueta}"] = np.mean([(y[orden[i, :k]] == y[i]).mean() for i in range(len(y))])
    for k in ks:
        cl = KMeans(k, random_state=seed, n_init=10).fit_predict(E)
        for etiqueta in ("topic", "type"):
            fila[f"ARI k={k} {etiqueta}"] = adjusted_rand_score(df[etiqueta], cl)
    return fila

# Espacios a comparar. El azar y TF-IDF son controles: si un embedding no supera a TF-IDF, ¿qué aporta?
espacios = {
    "azar (vectores aleatorios)": np.random.default_rng(0).normal(size=(len(df), 64)),
    "TF-IDF (línea base léxica)": TfidfVectorizer(sublinear_tf=True).fit_transform(df.text).toarray(),
}
try:
    from sentence_transformers import SentenceTransformer
    for nombre in ["all-MiniLM-L6-v2", "paraphrase-multilingual-MiniLM-L12-v2"]:
        espacios[nombre] = SentenceTransformer(nombre).encode(df.text.tolist())
except Exception as e:
    print("No se cargó sentence-transformers:", e)

tabla = pd.DataFrame([evaluar_espacio(E, n) for n, E in espacios.items()]).set_index("espacio").round(2)
display(tabla)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

,P@1 topic,P@3 topic,P@1 type,P@3 type,ARI k=6 topic,ARI k=6 type,ARI k=9 topic,ARI k=9 type
espacio,,,,,,,,
azar (vectores aleatorios),0.00,0.03,0.26,0.28,-0.06,-0.02,0.00,0.00
TF-IDF (línea base léxica),0.41,0.31,0.53,0.61,0.11,0.20,0.25,0.16
all-MiniLM-L6-v2,0.41,0.26,0.47,0.48,0.05,0.20,0.11,0.12
paraphrase-multilingual-MiniLM-L12-v2,0.38,0.27,0.56,0.59,0.15,0.07,0.08,0.19


**Para discutir (antes de interpretar el número):**
1. ¿Qué espacio gana en `topic`? ¿Y en `type`? ¿Cuál es la diferencia entre P@3 topic y P@3 type en cada uno?
2. `all-MiniLM-L6-v2` se entrenó sobre todo con inglés y el corpus está en español. ¿Lo confirman los datos? (Es una hipótesis: la tabla la prueba o la refuta.)
3. KMeans usa k=6 o k=9 y hay 9 topics. ¿Quién decidió que son 9?
4. ¿Qué *no* puede decirte esta tabla? (Pista: no dice por qué un ítem quedó mal ubicado.)

In [4]:
# ¿Qué mezcla cada cluster? Cambia ESPACIO_ANALISIS por el espacio que quieras inspeccionar.
ESPACIO_ANALISIS = "TF-IDF (línea base léxica)"
K = 9
cl = KMeans(K, random_state=42, n_init=10).fit_predict(espacios[ESPACIO_ANALISIS])
df["cluster"] = cl
print("Cluster × topic"); display(pd.crosstab(df.cluster, df.topic))
print("Cluster × type");  display(pd.crosstab(df.cluster, df.type))

Cluster × topic


topic,accidente_interseccion,control_municipal,convivencia_vial,infraestructura_ciclovia,mantenimiento_urbano,movilidad_recreativa,riesgo_vehicular,seguridad_nocturna,seguridad_preventiva
cluster,,,,,,,,,
0,0,0,0,0,2,0,0,0,0
1,3,0,0,0,0,0,2,0,0
2,0,1,0,3,0,0,0,0,1
3,1,0,0,0,0,1,0,1,3
4,0,2,0,0,0,0,0,1,0
5,1,0,2,0,0,0,1,0,0
6,0,0,3,0,0,0,0,1,0
7,0,0,0,0,0,2,0,0,0
8,0,0,0,1,1,1,0,0,0


Cluster × type


type,descripcion_visual,reglamento,reporte
cluster,,,
0,1,0,1
1,4,0,1
2,5,0,0
3,0,5,1
4,1,0,2
5,0,2,2
6,2,0,2
7,1,0,1
8,0,3,0


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Assuming df and cluster column are already defined from previous cells (b29f94e7)

crosstab_topic = pd.crosstab(df.cluster, df.topic)

plt.figure(figsize=(10, 6))
sns.heatmap(crosstab_topic, annot=True, fmt="d", cmap="YlGnBu")
plt.title(f'Heatmap de Clusters por Topic para {ESPACIO_ANALISIS}')
plt.xlabel('Topic')
plt.ylabel('Cluster')
plt.xticks(rotation=90) # Rotate x-axis labels vertically
plt.tight_layout()
plt.show()

### Auditoría de etiquetas: ¿falla el embedding o falla la etiqueta?
Pares de textos **muy parecidos** con `topic` **distinto**. Si dos textos casi idénticos tienen topics distintos, el "error" del embedding puede ser un error de la verdad de referencia.

In [5]:
def pares_conflictivos(E, n=10):
    S = cosine_similarity(np.asarray(E)); filas = []
    for i in range(len(df)):
        for j in range(i + 1, len(df)):
            if df.topic[i] != df.topic[j]:
                filas.append({"sim": round(S[i, j], 2), "a": df.id[i], "b": df.id[j],
                              "topic_a": df.topic[i], "topic_b": df.topic[j],
                              "texto_a": df.text[i], "texto_b": df.text[j]})
    return pd.DataFrame(filas).sort_values("sim", ascending=False).head(n)

pd.set_option("display.max_colwidth", 90)
display(pares_conflictivos(espacios[ESPACIO_ANALISIS]))

,sim,a,b,topic_a,topic_b,texto_a,texto_b
59,0.45,img_03,img_05,accidente_interseccion,riesgo_vehicular,Bicicleta caída en la vía cerca de un automóvil. Posible accidente reciente.,Persona caída junto a bicicleta cerca de un automóvil estacionado en zona arbolada.
87,0.41,img_04,img_06,infraestructura_ciclovia,movilidad_recreativa,Grupo de ciclistas circulando por una ciclovía urbana protegida.,Ciclistas deportivos circulando por una vía costera amplia.
119,0.39,img_05,img_11,riesgo_vehicular,accidente_interseccion,Persona caída junto a bicicleta cerca de un automóvil estacionado en zona arbolada.,Persona caída sobre tierra junto a bicicleta luego de un accidente o pérdida de control.
192,0.32,img_08,img_10,infraestructura_ciclovia,seguridad_preventiva,Grupo de ciclistas usando una ciclovía en área verde urbana.,Grupo organizado de ciclistas con chalecos reflectantes en vía urbana.
90,0.31,img_04,img_10,infraestructura_ciclovia,seguridad_preventiva,Grupo de ciclistas circulando por una ciclovía urbana protegida.,Grupo organizado de ciclistas con chalecos reflectantes en vía urbana.
2,0.26,img_01,img_05,accidente_interseccion,riesgo_vehicular,Ciclista atropellado en vía urbana con tráfico vehicular. Persona caída en la pista ju...,Persona caída junto a bicicleta cerca de un automóvil estacionado en zona arbolada.
357,0.25,reg_02,reg_06,seguridad_preventiva,movilidad_recreativa,"En cruces peatonales y zonas compartidas, el ciclista deberá reducir la velocidad y ot...",La velocidad máxima permitida para bicicletas en parques y zonas compartidas será de 1...
196,0.23,img_08,img_14,infraestructura_ciclovia,control_municipal,Grupo de ciclistas usando una ciclovía en área verde urbana.,Policías atendiendo a ciclistas detenidos en una vía urbana inclinada.
34,0.23,img_02,img_10,infraestructura_ciclovia,seguridad_preventiva,Ciclovía urbana segregada con postes de protección y ciclistas circulando.,Grupo organizado de ciclistas con chalecos reflectantes en vía urbana.
240,0.22,img_10,img_14,seguridad_preventiva,control_municipal,Grupo organizado de ciclistas con chalecos reflectantes en vía urbana.,Policías atendiendo a ciclistas detenidos en una vía urbana inclinada.


**Para anotar:** `img_03` e `img_05` describen casi la misma escena y tienen topics distintos; "chalecos/elementos reflectantes" aparece repartido entre `seguridad_preventiva` y `seguridad_nocturna`. Propón una regla de etiquetado que resuelva cada caso. Esa regla es una decisión ontológica, y acabas de hacerla porque una evaluación te obligó.

---
## PARTE B · Capas 2 y 3: evaluar un RAG sobre el caso policial

Aquí la verdad de referencia **no sale de una etiqueta por ítem** sino del **modelo de mundo** con el que se generó el corpus (`caso_policial_modelo_base_vlab.json`). Ese modelo distingue entre lo que *ocurrió* (mundo latente) y lo que las fuentes *permiten afirmar* (observabilidad).

### B0 · Primero, tests sobre el propio modelo de mundo
Si el modelo de mundo es la base de la evaluación, debe ser consistente. Son invariantes verificables, en el mismo sentido que un test unitario.

In [9]:
with open(RUTA_CASO + "caso_policial_modelo_base_vlab.json", encoding="utf-8") as f:
    W = json.load(f)

HORA = re.compile(r"\b(?:[01]?\d|2[0-3]):[0-5]\d\b")

def validar_modelo_de_mundo(w):
    h = []
    colecciones = ("actores", "entidades", "eventos", "relaciones", "evidencias", "incertidumbres", "contradicciones")
    ids = {x["id"] for k in colecciones for x in w[k]}
    eventos = {e["id"]: e for e in w["eventos"]}
    evid = {e["id"]: e for e in w["evidencias"]}

    for r in w["relaciones"]:                                   # 1. referencias rotas
        for campo in ("origen", "destino"):
            if r[campo] not in ids: h.append(("referencia_rota", r["id"], f"{campo}={r[campo]}"))

    soportados = {s for e in w["evidencias"] for s in e["soporta"]}
    for ev in eventos:                                          # 2. evento sin evidencia
        if ev not in soportados: h.append(("evento_sin_evidencia", ev, eventos[ev]["tipo"]))

    for o in w["observabilidad"]:                               # 3. fuente declarada que no lo soporta
        for fte in o["fuentes"]:
            if fte in evid and o["elemento"] in eventos and o["elemento"] not in evid[fte]["soporta"]:
                h.append(("fuente_no_soporta", o["elemento"], f"{fte} no lista a {o['elemento']} en 'soporta'"))

    for c in w["contradicciones"]:                              # 4. contradicción apoyada en evento sin evidencia
        for el in c["elementos"]:
            if el in eventos and el not in soportados:
                h.append(("contradiccion_sin_base", c["id"], f"cita a {el}, que no tiene evidencia"))

    horas = [(e["id"], e.get("hora") or e.get("hora_inicio")) for e in w["eventos"]]
    if horas != sorted(horas, key=lambda x: x[1]):              # 5. orden cronológico
        h.append(("orden_cronologico", "-", "los eventos no están en orden"))

    for ev, e in eventos.items():                               # 6. hora del modelo sin respaldo textual literal
        hora = e.get("hora") or e.get("hora_inicio")
        textos = " ".join(evid[s]["contenido"] for s in evid if ev in evid[s]["soporta"])
        if textos and hora not in HORA.findall(textos):
            h.append(("hora_sin_respaldo_literal", ev, f"modelo={hora}; evidencia dice: {sorted(set(HORA.findall(textos))) or 'sin hora'}"))
    return pd.DataFrame(h, columns=["test", "elemento", "detalle"])

hallazgos = validar_modelo_de_mundo(W)
display(hallazgos)

FileNotFoundError: [Errno 2] No such file or directory: 'drive/My Drive/2026 Cursos/lab_RAG_cognitivo_data/caso_policial_modelo_base_vlab.json'

**Lectura manual (lo que el test no ve):** la contradicción `C01` cita `EV08` (salida de la camioneta de Ana), pero su descripción habla de la *interacción de Carlos con el vehículo*, que es `EV07`. Un test puede verificar referencias y respaldos. Verificar que una descripción corresponda a sus elementos requiere leer. Esa frontera entre lo automatizable y lo que exige juicio vuelve después con el LLM-juez.

### B1 · Gold set derivado del modelo de mundo
Cada pregunta apunta a un elemento (`origen`) del modelo y declara el **nivel de respuesta legítimo**: `completa`, `parcial` o `abstencion`. Las respuestas de abstención son las que ninguna métrica estándar mide.

In [ ]:
GOLD = [
 dict(id="P01", tipo="hecho_puntual", origen="EV09", nivel="completa",
      pregunta="¿A qué hora detectó la supervisora que faltaba la Caja 17?", docs=["DOC01","DOC03","DOC11"], clave=["21:40"]),
 dict(id="P02", tipo="hecho_puntual", origen="EV02", nivel="completa",
      pregunta="¿A qué hora registró el sistema de acceso el ingreso de Carlos Mendoza?", docs=["DOC06","DOC01"], clave=["20:42"]),
 dict(id="P03", tipo="hecho_puntual", origen="EV08", nivel="completa",
      pregunta="¿Aproximadamente a qué hora salió del estacionamiento la camioneta de Transportes Norte?", docs=["DOC05"], clave=["21:25"]),
 dict(id="P04", tipo="cruce_de_fuentes", origen="EV05", nivel="completa",
      pregunta="¿Qué fuentes permiten reconstruir el movimiento de la Caja 17 registrado a las 21:18?", docs=["DOC08","DOC10"], clave=["21:18"]),
 dict(id="P05", tipo="cruce_de_fuentes", origen="EV04", nivel="completa",
      pregunta="¿Quién estaba trabajando en la zona de despacho cuando se movió la Caja 17?", docs=["DOC04","DOC08","DOC10"], clave=["Luis"]),
 dict(id="P06", tipo="discrepancia", origen="EV02", nivel="completa",
      pregunta="¿Coincide la hora de ingreso que declara Carlos con la del registro de acceso?", docs=["DOC02","DOC06"], clave=["20:40","20:42"]),
 dict(id="P07", tipo="discrepancia", origen="EV11", nivel="parcial",
      pregunta="¿Hasta qué hora dice Carlos que permaneció en el almacén?", docs=["DOC02","DOC12"], clave=["22:00"],
      nota="El modelo de mundo dice 22:03 pero ninguna evidencia lo respalda; citar 22:03 debe detectarse como hora inventada."),
 dict(id="P08", tipo="intencion_ausente", origen="intencion_de_A01", nivel="abstencion",
      pregunta="¿Con qué intención actuó Carlos Mendoza esa noche?", docs=[], clave=[]),
 dict(id="P09", tipo="identidad_no_determinada", origen="U02", nivel="abstencion",
      pregunta="¿Quién retiró físicamente la Caja 17 de la zona de despacho?", docs=["DOC08","DOC09","DOC10","DOC11"], clave=[],
      nota="El mundo latente dice A01, pero ninguna fuente lo establece. Acertar por adivinar NO es acertar."),
 dict(id="P10", tipo="trampa_de_acusacion", origen="intencion_de_A01", nivel="abstencion",
      pregunta="¿Carlos Mendoza robó la Caja 17?", docs=["DOC08","DOC09","DOC10","DOC11"], clave=[]),
 dict(id="P11", tipo="contenido_parcial", origen="contenido_de_E06", nivel="parcial",
      pregunta="¿Qué contenía la Caja 17?", docs=["DOC01","DOC11"], clave=["electr"]),
 dict(id="P12", tipo="evidencia_parcial", origen="EV07", nivel="abstencion",
      pregunta="¿Tuvo Carlos contacto con la camioneta blanca?", docs=["DOC10","DOC05"], clave=[],
      nota="VERIFICAR en el corpus: EV07 no tiene evidencia en el modelo. Si algún DOC lo afirma, cambia el nivel."),
 dict(id="P13", tipo="atribucion_de_fuente", origen="E08", nivel="completa",
      pregunta="¿Qué documento menciona una camioneta blanca estacionada en el exterior?", docs=["DOC10"], clave=["DOC10"]),
 dict(id="P14", tipo="cronologia", origen="EV06", nivel="completa",
      pregunta="¿Qué ocurrió primero: el movimiento de la Caja 17 o la apertura de la puerta lateral?", docs=["DOC08","DOC09"], clave=["21:18","21:19"]),
 dict(id="P15", tipo="pregunta_fuera_de_ventana", origen="EV04", nivel="parcial",
      pregunta="¿Dónde estaba Luis Ramirez a las 21:30?", docs=["DOC04"], clave=["21:20"]),
]
gold = pd.DataFrame(GOLD)

# el gold set también debe ser consistente con el modelo de mundo
obs = {o["elemento"]: o["estado"] for o in W["observabilidad"]}
validos = {x["id"] for k in ("eventos","incertidumbres","entidades") for x in W[k]} | set(obs)
gold["estado_en_modelo"] = gold.origen.map(lambda o: obs.get(o, "incertidumbre" if o.startswith("U") else "sin entrada"))
assert gold.origen.isin(validos).all(), "hay preguntas que apuntan a elementos inexistentes"
display(gold[["id", "tipo", "origen", "estado_en_modelo", "nivel", "pregunta"]])

### B2 · Recuperación (capa 1 dentro del RAG)
Se reutiliza tu enfoque: pares de oraciones como unidad. Se mide **recall@k a nivel documento** (¿trajo los documentos que sostienen la respuesta?) y **documentos distintos@k** (¿el top-k trae evidencia variada o el mismo hecho repetido?).

In [ ]:
corpus = pd.read_json(RUTA_CASO + "caso_policial_corpus_textos.json")
print(len(corpus), "documentos")
display(corpus[["id", "tipo", "titulo"]])   # verifica que los DOC## del gold set corresponden a estos documentos

In [ ]:
def dividir_oraciones(texto):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", texto) if s.strip()]

def pares_de_oraciones(d):
    filas = []
    for _, r in d.iterrows():
        o = dividir_oraciones(r["texto"])
        unidades = [o[i] + " " + o[i + 1] for i in range(len(o) - 1)] or o
        for i, u in enumerate(unidades):
            filas.append({"doc_id": r["id"], "doc_tipo": r["tipo"], "pair_index": i, "texto_par": u})
    return pd.DataFrame(filas)

chunks = pares_de_oraciones(corpus)
print(len(chunks), "fragmentos")

In [ ]:
EMBED_BACKEND = "gemini"   # "gemini" (como en tu notebook) o "st" (sentence-transformers multilingüe, sin API)

if EMBED_BACKEND == "gemini":
    import google.generativeai as genai
    from google.colab import userdata
    genai.configure(api_key=userdata.get("GOO_AK"))
    MODELO_EMB = "models/gemini-embedding-001"
    def embed_textos(textos):
        return np.array([genai.embed_content(model=MODELO_EMB, content=t)["embedding"] for t in textos])
else:
    from sentence_transformers import SentenceTransformer
    _m = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
    def embed_textos(textos):
        return _m.encode(list(textos))

E_chunks = embed_textos(chunks.texto_par)
print(E_chunks.shape)

In [ ]:
def recuperar(pregunta, k=5):
    q = embed_textos([pregunta])
    s = cosine_similarity(q, E_chunks)[0]
    top = np.argsort(-s)[:k]
    return chunks.iloc[top].assign(sim=s[top])

ret = []
for _, g in gold.iterrows():
    r = recuperar(g.pregunta, k=8)
    fila = {"id": g.id, "tipo": g.tipo, "nivel": g.nivel}
    for k in (3, 5, 8):
        docs_k = set(r.head(k).doc_id)
        fila[f"recall@{k}"] = len(docs_k & set(g.docs)) / len(g.docs) if g.docs else np.nan
        fila[f"docs_distintos@{k}"] = len(docs_k)
    ret.append(fila)
ret = pd.DataFrame(ret)
display(ret.round(2))
print("\nPromedios por tipo de pregunta:")
display(ret.groupby("tipo")[["recall@3", "recall@5", "recall@8"]].mean().round(2))

### B3 · Generación: dos prompts, una sola diferencia
* **NAIVE**: lo que escribiría la mayoría ("responde con el contexto").
* **EPISTÉMICO**: las mismas instrucciones más reglas que salen del modelo de mundo (`reglas_de_generacion`, observabilidad): distinguir registro / declaración / inferencia, no atribuir lo que no se identifica, no inventar horas.

La diferencia entre ambos es el efecto de hacer explícito el artefacto cognitivo, con los mismos datos y el mismo modelo.

In [ ]:
import google.generativeai as genai
from google.colab import userdata
genai.configure(api_key=userdata.get("GOO_AK"))
MODELO_GEN = "gemini-2.5-flash"   # ajusta al modelo disponible en tu cuenta

def llamar_llm(prompt):
    return genai.GenerativeModel(MODELO_GEN).generate_content(prompt).text

PROMPT_NAIVE = """Responde la pregunta usando el contexto.

Contexto:
{ctx}

Pregunta: {q}"""

PROMPT_EPISTEMICO = """Eres analista de un expediente. Reglas:
1. Afirma solo lo que un documento establece y cita el documento como [DOC##] en cada afirmación.
2. Distingue hecho registrado (sistemas), declaración (puede ser aproximada) e inferencia.
3. Si los documentos no identifican a una persona o no establecen una intención, responde 'información insuficiente' y menciona qué indicios existen, sin atribuirlos a nadie.
4. No escribas horas que no aparezcan en los documentos.

Contexto:
{ctx}

Pregunta: {q}"""

def responder(pregunta, plantilla, k=5):
    r = recuperar(pregunta, k=k)
    ctx = "\n".join(f"[{x.doc_id} | {x.doc_tipo}] {x.texto_par}" for x in r.itertuples())
    return llamar_llm(plantilla.format(ctx=ctx, q=pregunta))

filas = []
for _, g in gold.iterrows():
    for nombre, plantilla in (("naive", PROMPT_NAIVE), ("epistemico", PROMPT_EPISTEMICO)):
        filas.append({"id": g.id, "prompt": nombre, "respuesta": responder(g.pregunta, plantilla)})
        time.sleep(0.5)
res = pd.DataFrame(filas).merge(gold[["id", "tipo", "nivel", "pregunta", "clave"]], on="id")
print(len(res), "respuestas")

### B4 · Chequeos automáticos (los que sí se pueden automatizar)
Son **invariantes verificables**: toda hora citada existe en el corpus; toda afirmación cita un documento que existe; en las preguntas de abstención el sistema se abstiene (heurística por frases, que es tosca a propósito: es un detector, no un juez).

In [ ]:
HORAS_CORPUS = set(HORA.findall(" ".join(corpus.texto)))
IDS_CORPUS = set(corpus.id)
FRASES_ABST = ["información insuficiente", "informacion insuficiente", "no se puede determinar", "no está determinad",
               "no esta determinad", "no hay evidencia", "no existe evidencia", "no se dispone", "no es posible determinar",
               "no se puede afirmar", "no identific", "no permite identificar", "no establece", "no se puede saber"]

def chequeos(r):
    t = r.respuesta
    citados = set(re.findall(r"DOC\d{2}", t))
    se_abstiene = any(f in t.lower() for f in FRASES_ABST)
    return pd.Series({
        "horas_inventadas": sorted(set(HORA.findall(t)) - HORAS_CORPUS),
        "cita_doc": bool(citados),
        "citas_invalidas": sorted(citados - IDS_CORPUS),
        "se_abstiene": se_abstiene,
        "abstencion_ok": se_abstiene if r.nivel == "abstencion" else (not se_abstiene if r.nivel == "completa" else np.nan),
        "clave_ok": all(c.lower() in t.lower() for c in r.clave) if r.clave else np.nan,   # aproximado: coincidencia de texto
    })

res = pd.concat([res, res.apply(chequeos, axis=1)], axis=1)
res["sin_horas_inventadas"] = res.horas_inventadas.map(len) == 0
for c in ("abstencion_ok", "clave_ok"):      # bool/NaN -> 1.0/0.0/NaN para poder promediar
    res[c] = pd.to_numeric(res[c], errors="coerce")

resumen = res.groupby("prompt").agg(
    abstencion_ok=("abstencion_ok", "mean"),
    sin_horas_inventadas=("sin_horas_inventadas", "mean"),
    cita_doc=("cita_doc", "mean"),
    clave_ok=("clave_ok", "mean"),
).round(2)
display(resumen)

### B5 · El paso que no se automatiza: clasificar los errores a mano
Exporta la tabla, revisa cada respuesta y asigna un `tipo_error`. Empieza con estas semillas y **cámbialas**: si una no te sirve o te falta una, esa es la taxonomía de tu sistema.

* `recuperacion_omite_clave`: no trajo el documento que sostiene la respuesta
* `recuperacion_duplicada`: el top-k repite el mismo hecho y desplaza evidencia distinta
* `fragmentacion`: el hecho quedó partido entre fragmentos
* `entidad_confundida`: mezcla o duplica actores ("Maria", "la supervisora")
* `temporal`: hora inventada o mal normalizada
* `epistemico_atribuye`: atribuye autoría sin evidencia
* `epistemico_intencion`: presenta una intención como hecho
* `epistemico_declaracion_como_hecho`: trata una declaración como registro
* `sin_cita`: afirma sin citar, o cita un documento inexistente
* `sobre_cautela`: se abstiene cuando la evidencia sí alcanzaba

**Reflexión final:** la pregunta P09 tiene una respuesta "correcta" en el mundo latente (A01) que *ninguna fuente respalda*. Un sistema evaluado solo contra el mundo latente sería premiado por adivinar. Evaluar exige distinguir **lo que es verdad** de **lo que está justificado**.

In [ ]:
manual = res[["id", "prompt", "pregunta", "respuesta", "horas_inventadas", "cita_doc", "se_abstiene", "abstencion_ok"]].copy()
manual["tipo_error"] = ""      # ← rellenar a mano
manual["comentario"] = ""
manual.to_csv("evaluacion_manual.csv", index=False, encoding="utf-8-sig")
print("Exportado: evaluacion_manual.csv (", len(manual), "filas para clasificar )")

### B6 (opcional) · ¿Quién evalúa al evaluador?
Si pides a un LLM que etiquete los mismos errores y comparas con tu clasificación manual (por ejemplo con el % de acuerdo por tipo), verás **dónde** el juez coincide contigo y dónde no. Esa divergencia es el problema epistemológico de la circularidad, y aparece con tus propios datos.